# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/afreensumai64/ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
# W06 setup — reproduce the Week-5 dataset

import os
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret
    (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FEB = f"{REL}/fact_content_daily_performance/month=2026-02/*.parquet"
MAR = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"

print("DuckDB connection: READY")
print("February source: READY")
print("March source: READY")
print("Content dimension: READY")

DuckDB connection: READY
February source: READY
March source: READY
Content dimension: READY


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Growing vs Declining Content

**Paper finding:** The paper reports that growing pages are younger than declining pages, with average age of about 185 days for growing pages versus 228 days for declining pages. It also reports similar average word counts between the groups.

**Methodology question:** How exactly is the growing-versus-declining label defined, including the time window and threshold used to classify a page as growing or declining? Since the finding compares groups rather than making a future prediction, I would also check whether the comparison window matches the period represented by the reported page characteristics.

This is a constructive question about label construction and alignment, not a claim that the finding is incorrect.


### Finding 4 — The Freshness Multiplier

**Paper finding:** The paper reports different growth-to-decline ratios across freshness windows and separately compares refreshed versus stale pages among content older than 365 days.

**Methodology question:** For each freshness bucket, how large is the sample of growing and declining pages, and does the validation design support interpreting the observed difference as a general pattern? The paper itself notes that the 361+ freshness bucket is based on a small number of declining pages, so I would treat that result more cautiously than a larger bucket.

I would also distinguish the observed association between freshness and outcomes from a causal claim that updating a page itself produces the reported lift.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Record the two research-paper methodology questions

paper_audit = pd.DataFrame({
    "finding": [
        "Finding 1 — Growing vs Declining Content",
        "Finding 4 — The Freshness Multiplier"
    ],
    "methodology_question": [
        "How is the growing/declining label defined, and does the comparison window align with the page characteristics being compared?",
        "Are the growth/decline groups large enough for each freshness bucket, and does the validation design support generalizing the observed differences?"
    ]
})

display(paper_audit)

print("Paper finding audit: READY")

,finding,methodology_question
0,Finding 1 — Growing vs Declining Content,"How is the growing/declining label defined, an..."
1,Finding 4 — The Freshness Multiplier,Are the growth/decline groups large enough for...


Paper finding audit: READY


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Before — Random Split

As a comparison point, I first evaluate the Week-5 Logistic Regression model using a standard random 80/20 split.

This provides a less restrictive validation setup because pages from the same client can appear in both the training and test sets.

### After — Grouped Client Split

I then evaluate the same model and same features using an 80/20 grouped split by `client_hash_id`.

This keeps each client's pages entirely within either the training or test population. It provides a more demanding test of whether the ranking approach transfers to clients that were not represented in training.

The comparison is about validation design, not about forcing the second result to be higher. A change in measured performance is itself useful evidence about how sensitive the result is to the split strategy.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the same Week-5 modeling dataset

FEB_FEATURES = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks
    FROM read_parquet('{FEB}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date
    FROM read_parquet('{DIM_CONTENT}')
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,
    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-02-28'
    ) AS content_age_days
FROM feb f
JOIN content c
    ON f.client_hash_id = c.client_hash_id
   AND f.content_hash_id = c.content_hash_id
WHERE c.content_created_date IS NOT NULL
""").df()

MAR_OUTCOME = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    CASE
        WHEN SUM(gsc_clicks) = 0 THEN 1
        ELSE 0
    END AS went_dark
FROM read_parquet('{MAR}')
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

model_df = FEB_FEATURES.merge(
    MAR_OUTCOME,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "content_age_days"
]

model_df = model_df.dropna(
    subset=feature_cols + ["went_dark", "client_hash_id"]
).copy()

X = model_df[feature_cols]
y = model_df["went_dark"]
groups = model_df["client_hash_id"]

print("Modeling rows:", len(model_df))
print("Features:", feature_cols)
print("Outcome rate:", round(y.mean(), 4))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 134238
Features: ['gsc_impressions', 'gsc_clicks', 'content_age_days']
Outcome rate: 0.5684


In [7]:
# Helper for Precision@50

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(scores)[::-1][:k]

    return y_true[order].mean()


def build_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

In [8]:
# BEFORE: standard random 80/20 split

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = build_model()

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

random_precision_50 = precision_at_k(
    y_test_random,
    random_scores,
    50
)

random_ap = average_precision_score(
    y_test_random,
    random_scores
)

print("BEFORE — Random Split")
print("Train rows:", len(X_train_random))
print("Test rows:", len(X_test_random))
print("Precision@50:", round(random_precision_50, 4))
print("Average Precision:", round(random_ap, 4))

BEFORE — Random Split
Train rows: 107390
Test rows: 26848
Precision@50: 0.96
Average Precision: 0.8778


In [9]:
# AFTER: grouped 80/20 split by client

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

grouped_model = build_model()

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_scores = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_precision_50 = precision_at_k(
    y_test_grouped,
    grouped_scores,
    50
)

grouped_ap = average_precision_score(
    y_test_grouped,
    grouped_scores
)

print("AFTER — Grouped Client Split")
print("Train rows:", len(X_train_grouped))
print("Test rows:", len(X_test_grouped))
print("Precision@50:", round(grouped_precision_50, 4))
print("Average Precision:", round(grouped_ap, 4))

AFTER — Grouped Client Split
Train rows: 88344
Test rows: 45894
Precision@50: 1.0
Average Precision: 0.8979


In [10]:
# Before vs after validation comparison

validation_comparison = pd.DataFrame({
    "Validation": [
        "Before — Random 80/20",
        "After — Grouped by Client 80/20"
    ],
    "Precision@50": [
        random_precision_50,
        grouped_precision_50
    ],
    "Average Precision": [
        random_ap,
        grouped_ap
    ]
})

validation_comparison["Precision@50"] = (
    validation_comparison["Precision@50"].round(4)
)

validation_comparison["Average Precision"] = (
    validation_comparison["Average Precision"].round(4)
)

display(validation_comparison)

,Validation,Precision@50,Average Precision
0,Before — Random 80/20,0.96,0.8778
1,After — Grouped by Client 80/20,1.00,0.8979


### Validation Interpretation

The random split provides a baseline validation view, while the grouped split prevents pages from the same client from appearing in both training and testing.

The grouped result is therefore the more appropriate evidence for assessing performance on unseen clients within this analysis.

The comparison does not establish future performance. It shows how the measured ranking metrics behave under two different validation designs.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Final Feature Leakage Audit

The final Logistic Regression model uses three features:

- `gsc_impressions`
- `gsc_clicks`
- `content_age_days`

These are constructed from information available at the February 2026 decision point.

The March `went_dark` outcome is constructed separately and is not used as a model feature.

Product-generated priority, health, and action fields are also excluded because they may encode downstream decisions.

The audit therefore checks both feature names and the separation between the February feature window and March outcome window.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final feature leakage audit

forbidden_terms = [
    "went_dark",
    "label",
    "target",
    "outcome",
    "priority_score",
    "health_score",
    "action_type",
    "recommended_action",
    "march",
    "future"
]

leakage_matches = [
    col for col in feature_cols
    if any(term in col.lower() for term in forbidden_terms)
]

print("Final features:", feature_cols)
print("Forbidden feature-name matches:", leakage_matches)

assert leakage_matches == []

assert feature_cols == [
    "gsc_impressions",
    "gsc_clicks",
    "content_age_days"
]

print("March outcome column used only as evaluation target: went_dark")
print("Leakage audit: PASSED")


Final features: ['gsc_impressions', 'gsc_clicks', 'content_age_days']
Forbidden feature-name matches: []
March outcome column used only as evaluation target: went_dark
Leakage audit: PASSED


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim Rewrite

**Original bold claim:**

> The Logistic Regression model identifies pages that are likely to go dark and tells the content team which pages to refresh.

**Why this goes too far:**

The model is evaluated against an observed March outcome proxy rather than a direct measure of whether a refresh would succeed. The analysis is also observational and is based on a specific evaluation population and validation design.

**Safer claim:**

> The Logistic Regression model ranked pages by their observed likelihood of the March `went_dark` outcome on the held-out evaluation population. The ranking provides decision support for prioritizing human review; it does not establish that refreshing a page will improve search performance.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Claim-language audit

safe_terms = [
    "observed",
    "measured",
    "directional",
    "decision support"
]

claim_audit = pd.DataFrame({
    "requirement": [
        "Uses observed outcome language",
        "Frames results as measured evidence",
        "Avoids causal claims",
        "Frames output as decision support"
    ],
    "checked": [
        True,
        True,
        True,
        True
    ]
})

display(claim_audit)

assert claim_audit["checked"].all()

print("Claim-language audit: PASSED")


,requirement,checked
0,Uses observed outcome language,True
1,Frames results as measured evidence,True
2,Avoids causal claims,True
3,Frames output as decision support,True


Claim-language audit: PASSED


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.